# B4 · Coordinate reference systems in practice: measuring Nepal in metres

> **Time:** about 100 minutes  
> **You need:** Lessons B1 to B3. You should be able to read a layer with `gpd.read_file` and make points with `gpd.points_from_xy`.  
> **By the end you can:**
> 1. tell a **geographic** CRS from a **projected** one, and read a CRS's name, EPSG code and units,
> 2. convert layers with `to_crs`, label layers with `set_crs`, and recognise the classic mix-up between the two,
> 3. measure areas, lengths and buffers in metres and km in UTM zone 45N, and choose a sensible CRS for work in Nepal.

Since Lesson B1 you have met the same warning in many forms: degrees are not distances. In Lesson B2 shapely measured the Kathmandu Valley box in "square degrees", and in Lesson B3 every layer said `EPSG:4326`. This lesson explains what that code means, and how to convert your layers into **metres**, so that areas, lengths and buffers come out in km² and km.

**Counting lines:** `fig, ax = plt.subplots()` and `plt.show()` do not count toward the Q1/Q2/Q3 line counts.

**How to practise:** after every step there are three questions. **Q1** needs one line of code, **Q2** two lines, **Q3** about three. Type your answer in the empty cell under each question, run it, then open **Show solution** to check. Try first, even if you are not sure: struggling for a few minutes is how the skill grows.

<!-- pygile-nav -->
**PyGILE Course** · lesson 14 of 21 · [Course home](../README.md)  
← Previous: [B3 · GeoDataFrames and file formats: Nepal's districts as a table](B03_GeoDataFrames_and_file_formats.ipynb) · Next: [B5 · Selecting by attribute: finding the districts you need](B05_Selecting_by_attribute.ipynb) →

## 1. Geographic and projected coordinate systems

A **coordinate reference system** (CRS) says what the numbers in your coordinates mean: where zero is, which way is positive, and in which unit. There are two families:

- A **geographic CRS** gives positions on the curved Earth as angles: longitude and latitude, in degrees. **WGS 84**, used by GPS, is one.
- A **projected CRS** flattens the Earth onto a plane, with x and y in **metres**. The flattening is called a **map projection**. Every projection distorts something: area, shape, distance or direction. Each one chooses what to keep.

Both kinds also fix a model of the Earth's shape: a slightly flattened sphere called an **ellipsoid**. How that model is tied to the real Earth is called the **datum**.

CRSs are known by numbers in the **EPSG** registry (named after the European Petroleum Survey Group, which started it). Four codes cover most work in Nepal:

| Code | Name | Type | Unit | Use it for |
| --- | --- | --- | --- | --- |
| EPSG:4326 | WGS 84 | geographic | degree | storing and sharing data, GPS |
| EPSG:32644 | WGS 84 / UTM zone 44N | projected | metre | measuring in western Nepal |
| EPSG:32645 | WGS 84 / UTM zone 45N | projected | metre | measuring anywhere in Nepal (this course's choice) |
| EPSG:3857 | WGS 84 / Pseudo-Mercator | projected | metre | background tiles of web maps; **never** for measuring |

The `pyproj` library knows every code: `pyproj.CRS("EPSG:32645")`.

In [ ]:
import math
from pathlib import Path

import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import pyproj
from shapely import LineString

DATA = Path("..") / "data"
OUT = Path("..") / "outputs"
OUT.mkdir(exist_ok=True)

districts = gpd.read_file(DATA / "nepal_districts.geojson")
peaks = pd.read_csv(DATA / "nepal_peaks.csv")
peaks_gdf = gpd.GeoDataFrame(peaks, geometry=gpd.points_from_xy(peaks["lon"], peaks["lat"]), crs="EPSG:4326")
trek = pd.read_csv(DATA / "everest_trek.csv")
trek_gdf = gpd.GeoDataFrame(trek, geometry=gpd.points_from_xy(trek["lon"], trek["lat"]), crs="EPSG:4326")

districts.crs

In [ ]:
for code in [4326, 32644, 32645, 3857]:
    crs = pyproj.CRS(code)
    print(code, crs.name, "| projected:", crs.is_projected, "| unit:", crs.axis_info[0].unit_name)

`districts.crs` describes EPSG:4326 in full: a geographic 2D CRS named WGS 84, with the datum "World Geodetic System 1984" and Greenwich as the prime meridian.

Look at the **Axis Info**: the official definition lists latitude first. GeoPandas and shapely ignore that order and always store **x = longitude**, as you have done since Lesson B1. Some other programs follow the official order, which is one reason why lat/lon swaps are so common.

The loop confirms the table: only 4326 is in degrees; the other three are projected and in metres.

### Practice

**Q1 · one line:** Is the CRS of `districts` projected? Use `.is_projected`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
districts.crs.is_projected
```

`False`: EPSG:4326 is geographic, so its coordinates are angles in degrees.
</details>

**Q2 · two lines:** Make the CRS for UTM zone 45N with `pyproj.CRS(32645)`, store it as `utm45`, and print its name and its EPSG code with `.to_epsg()`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
utm45 = pyproj.CRS(32645)
print(utm45.name, utm45.to_epsg())
```

It prints `WGS 84 / UTM zone 45N 32645`. The name tells you the datum (WGS 84) and the projection (UTM zone 45N).
</details>

**Q3 · three lines:** Loop over `utm45.axis_info` and print each axis's `name`, `direction` and `unit_name`.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
for axis in utm45.axis_info:
    print(axis.name, axis.direction, axis.unit_name)
```

`Easting east metre`, then `Northing north metre`. In a projected CRS, x is called **easting** and y **northing**, both in metres.
</details>

## 2. UTM zones, and choosing a CRS for Nepal

**UTM** (Universal Transverse Mercator) cuts the world into 60 north–south **zones**, each 6° of longitude wide. Each zone has its own projection, centred on its **central meridian**, so distortion inside a zone is small. The zone number is `int((lon + 180) / 6) + 1`.

Within a zone, the **easting** counts metres from a line 500,000 m west of the central meridian, so eastings inside the zone are never negative. The **northing** counts metres north of the equator.

Nepal spans **two zones**: zone 44N west of 84° E and zone 45N east of it. For one map of the whole country, this course uses **UTM 45N (EPSG:32645)**. It slightly stretches the far west, which lies outside the zone, by about 1% (you will measure this in the review). GeoPandas can suggest a zone with `gdf.estimate_utm_crs()`.

How to choose:

| Task | CRS |
| --- | --- |
| store, share, GPS, web | EPSG:4326 |
| measure distance, length, area or buffers anywhere in Nepal | EPSG:32645 (UTM 45N) |
| work only in western Nepal, west of 84° E | EPSG:32644 (UTM 44N) |
| precise national area totals | an **equal-area** projection (step 4) |
| background tiles on a web map | EPSG:3857, for display only |

The official topographic maps of Nepal's Survey Department use their own national grid, called Modified UTM (MUTM); you will meet it if you work with those map sheets.

In [ ]:
def utm_zone(lon_deg):
    """UTM zone number for a longitude in degrees."""
    return int((lon_deg + 180) / 6) + 1


for lon_deg in [80.1, 83.9, 84.0, 86.925, 88.2]:
    print(lon_deg, utm_zone(lon_deg))

In [ ]:
print(districts.estimate_utm_crs())
print(districts[districts["province_code"] == "NP07"].estimate_utm_crs())

Longitudes up to 83.9° fall in zone 44, and from 84.0° in zone 45. Everest (86.925° E) is in zone 45.

For the whole country, GeoPandas suggests EPSG:32645, because the middle of Nepal's bounding box is a little east of 84°. For the districts of NP07, in the far west, it suggests EPSG:32644.

### Practice

**Q1 · one line:** Which UTM zone holds the centre of the Kathmandu Valley box from Lesson B2, at 85.375° E?

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
utm_zone(85.375)
```

Zone 45. Most of central and eastern Nepal is in zone 45.
</details>

**Q2 · two lines:** Work out the central meridian of zone 45: its western edge is `(45 - 1) * 6 - 180` degrees, and the centre is 3° further east. Print both.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
west_edge_deg = (45 - 1) * 6 - 180
print(west_edge_deg, west_edge_deg + 3)
```

The zone starts at 84° E and is centred on 87° E. Everest, at 86.925° E, is very close to the centre, so its easting will be close to 500,000 m.
</details>

**Q3 · three lines:** Count how many of the eight peaks are in each zone. Work out a zone for every row at once with whole-number division (Lesson A2): `(peaks["lon"] + 180) // 6 + 1`. Then use `value_counts()`.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
zones = (peaks["lon"] + 180) // 6 + 1
print(zones.value_counts())
```

6 peaks are in zone 45 and 2 in zone 44: Dhaulagiri I and Annapurna I, west of 84° E. Manaslu, at 84.56° E, is barely inside zone 45. `//` works on a whole column, so no loop is needed.
</details>

## 3. Converting with `to_crs`

`gdf.to_crs(32645)` returns a new layer with every coordinate **converted** into UTM zone 45N. This is called **reprojecting**. The original layer is not changed.

The numbers change completely: Everest's (86.925, 27.9881) in degrees becomes an easting and a northing in metres. The shapes also change a little, because each CRS flattens the Earth in its own way.

Reprojecting works because the layer already knows its CRS. GeoPandas needs both ends of the trip: where the numbers come from (`.crs`) and where they go.

**Predict:** Everest is a little west of the central meridian, 87° E. Will its easting be a little more, or a little less, than 500,000 m?

In [ ]:
districts_utm = districts.to_crs(32645)
peaks_utm = peaks_gdf.to_crs(32645)

print(districts_utm.crs.name)
print(districts_utm.total_bounds)
print(peaks_utm.geometry.head(3))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
districts.plot(ax=ax1, color="lightgrey", edgecolor="white")
ax1.set_title("EPSG:4326 (degrees)")
districts_utm.plot(ax=ax2, color="lightgrey", edgecolor="white")
ax2.set_title("EPSG:32645, UTM zone 45N (metres)")
ax2.tick_params(axis="x", labelrotation=45)
plt.show()

Everest's easting is 492,625 m, a little less than 500,000, and its northing 3,095,886 m: about 3,096 km north of the equator.

The western edge of Nepal has an easting of about −177,524 m. It is negative because the far west lies outside zone 45, beyond the zone's own 500,000 m margin. That is allowed, but it is where UTM 45N stretches the map most.

The two maps look almost the same, because the change of shape is small. The big change is in the numbers on the axes, which now count metres. The small `1e6` above the y-axis means "× 1,000,000": 3.1 on that axis is 3,100,000 m. `ax2.tick_params(axis="x", labelrotation=45)` turns the long x labels so they do not overlap.

### Practice

**Q1 · one line:** Convert `trek_gdf` to UTM zone 45N and show the geometry of the first stop, Lukla.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(trek_gdf.to_crs(32645).geometry.iloc[0])
```

Lukla is at an easting of about 473,367 m and a northing of about 3,062,515 m. Without `print`, Jupyter would draw the point as a small picture instead of showing its numbers.
</details>

**Q2 · two lines:** Convert `districts` to UTM zone 44N (EPSG:32644) and print its `total_bounds`. Compare the smallest easting with the zone 45N one.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
districts_44 = districts.to_crs(32644)
print(districts_44.total_bounds)
```

In zone 44N the western edge has an easting of about 408,230 m, positive this time, because the far west is inside zone 44. Same land, different numbers: a coordinate only means something together with its CRS.
</details>

**Q3 · three lines:** Using `peaks_utm`, print Everest's easting and northing in **km**, rounded to 1 decimal place.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
everest_utm = peaks_utm[peaks_utm["name"] == "Everest"].geometry.iloc[0]
print(round(everest_utm.x / 1000, 1), round(everest_utm.y / 1000, 1))
```

It prints `492.6 3095.9`. Everest is about 3,096 km north of the equator, and 7.4 km west of the zone's central meridian (500 − 492.6).
</details>

## 4. Areas in km²

In a projected CRS, `.area` gives square metres. Divide by 1,000,000 to get km².

In EPSG:4326, `.area` gives square degrees, which mean nothing on the ground (Lesson B2). GeoPandas now **warns** you when you do this.

Nepal's area, summed over the 77 districts in UTM 45N, comes out at about **148,017 km²**. This is a result *from this data*, not an official figure: the boundaries were simplified to about 300 m, and UTM 45N stretches the west a little.

For a precise national total, use an **equal-area** projection, which keeps every area true, at the cost of shapes. **Asia North Albers Equal Area Conic** (code `ESRI:102025`, from the registry of the company Esri) is one that covers Nepal.

**Predict:** Kathmandu district is about 0.038 square degrees. How many km² is that?

In [ ]:
kathmandu = districts[districts["district"] == "Kathmandu"]
print(kathmandu.area)

In [ ]:
print(kathmandu.to_crs(32645).area / 1000000)

nepal_km2 = districts_utm.area.sum() / 1000000
print(round(nepal_km2))

In [ ]:
nepal_equal_area_km2 = districts.to_crs("ESRI:102025").area.sum() / 1000000
print(round(nepal_equal_area_km2))

The first cell gave about 0.0379 square degrees, with a pink **UserWarning**: *Geometry is in a geographic CRS. Results from 'area' are likely incorrect.* A warning does not stop the code, so the wrong number still appears. Read warnings.

In UTM 45N, Kathmandu district is about 414.3 km², and the 77 districts add up to 148,017 km².

The equal-area projection gives 147,649 km², about 368 km² (0.25%) less: that is how much UTM 45N stretches Nepal as a whole. For most work this difference does not matter; for a national statistic it does.

### Practice

**Q1 · one line:** What is the area of Chitawan district in km², in UTM 45N? Use `districts_utm`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
districts_utm[districts_utm["district"] == "Chitawan"].area / 1000000
```

About 2,240.5 km². Remember the source spelling, Chitawan.
</details>

**Q2 · two lines:** Add a column `area_km2` to `districts_utm`, then show the `district` and `area_km2` of the five largest districts.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
districts_utm["area_km2"] = districts_utm.area / 1000000
districts_utm.sort_values("area_km2", ascending=False)[["district", "area_km2"]].head()
```

Dolpa (about 7,965 km²) and Humla (about 6,039 km²) are the largest, then Gorkha, Taplejung and Mustang: all mountain districts on the northern border.
</details>

**Q3 · three lines:** Compare Nepal's total area in UTM 44N with the total in UTM 45N (`nepal_km2`). Print both, rounded, and the difference in km².

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
nepal_44_km2 = districts.to_crs(32644).area.sum() / 1000000
print(round(nepal_44_km2), round(nepal_km2))
print(round(nepal_km2 - nepal_44_km2))
```

148,013 km² in zone 44N and 148,017 km² in zone 45N: only 4 km² apart. Each zone stretches the half of Nepal outside it, west or east, and the two errors nearly balance. The equal-area total is smaller than both.
</details>

## 5. Web Mercator distorts area

**Web Mercator** (EPSG:3857) is the projection behind most online maps. It keeps the shapes of small areas and keeps north straight up, which makes zooming on a screen pleasant.

The price is area. Web Mercator stretches distances in every direction by 1 ÷ cos(latitude), so areas grow by 1 ÷ cos²(latitude). Near the equator the effect is small; towards the poles it is enormous. That is why Greenland looks as large as Africa on a web map, as on the left below.

![The same world in Mercator (left) and in a cylindrical equal-area projection (right)](../images/mercator_vs_equal_area.png)

*Image: the pyGIS project by Michael Mann (GPL-3.0).*

Web Mercator coordinates are in metres, which makes the mistake tempting. **Never measure in EPSG:3857.** Use it only to draw on top of web map tiles.

**Predict:** at Kathmandu's latitude, about 27.7° N, by how much will Web Mercator inflate the area?

In [ ]:
print(1 / math.cos(math.radians(27.7)) ** 2)

kathmandu_utm_km2 = (kathmandu.to_crs(32645).area / 1000000).iloc[0]
kathmandu_web_km2 = (kathmandu.to_crs(3857).area / 1000000).iloc[0]
print(round(kathmandu_utm_km2, 1), round(kathmandu_web_km2, 1))
print(round(kathmandu_web_km2 / kathmandu_utm_km2, 3))

The rule predicts a factor of about 1.276. Measured on the district, Web Mercator gives about 530.9 km² instead of 414.3 km²: 1.281 times too large. The small difference arises because UTM itself is not perfectly true, but both say the same thing: **28% too much area**.

`.iloc[0]` takes the single value out of the one-row result, as in Lesson A9.

### Practice

**Q1 · one line:** What is Nepal's total area in Web Mercator, in km²? Round it.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
round(districts.to_crs(3857).area.sum() / 1000000)
```

About 191,084 km², roughly 43,000 km² more than the 148,017 km² from UTM 45N. That imaginary extra is larger than any one of Nepal's provinces (Review 2 measures them).
</details>

**Q2 · two lines:** Work out the inflation factor 1 ÷ cos²(latitude) for 26.5° N (the southern edge of Nepal) and for 30° N (the far north). Print both, rounded to 3 decimal places.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(round(1 / math.cos(math.radians(26.5)) ** 2, 3))
print(round(1 / math.cos(math.radians(30)) ** 2, 3))
```

1.249 in the south and 1.333 in the north. Even within one country, Web Mercator inflates the northern districts more than the southern ones, so it distorts comparisons between districts too.
</details>

**Q3 · three lines:** Check that on real districts. Print the ratio of the Web Mercator area to the UTM 45N area for Humla, in the far north-west, and for Jhapa, in the far south-east.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
for name in ["Humla", "Jhapa"]:
    one = districts[districts["district"] == name]
    print(name, (one.to_crs(3857).area / one.to_crs(32645).area).iloc[0])
```

Humla comes out about 1.33 times too large and Jhapa about 1.26 times: the farther north, the larger the error, as the rule predicts.
</details>

## 6. Lengths and distances in km

Lengths work like areas. After `to_crs(32645)`, `.length` gives metres, and `a.distance(b)` gives the distance between two geometries in metres. Divide by 1,000 for km.

In Lesson B1 you estimated distances on the trek with Pythagoras and the 111 km rule, and in Lesson B2 shapely measured the trek line in degrees. Now you can measure it properly.

To measure a single geometry with a CRS, put it in a GeoSeries: `gpd.GeoSeries([line], crs="EPSG:4326")`. Then convert it like any layer.

In [ ]:
trek_line = gpd.GeoSeries([LineString(trek[["lon", "lat"]].to_numpy())], crs="EPSG:4326")
trek_line_utm = trek_line.to_crs(32645)
print(round(trek_line_utm.length.iloc[0] / 1000, 2))

In [ ]:
trek_utm = trek_gdf.to_crs(32645)
lukla_utm = trek_utm.geometry.iloc[0]
base_camp_utm = trek_utm.geometry.iloc[-1]
print(round(lukla_utm.distance(base_camp_utm) / 1000, 2))

The trek line is 42.37 km long, and the straight line from Lukla to Base Camp is 37.07 km. Your Lesson B1 estimates, 42.4 km and 37.1 km, were very close: over a few tens of km, the flat-Earth shortcut works. Over hundreds of km, or for areas, it does not.

The real walking distance is longer still, because the trail winds between the stops. Eight stops are only a sketch of the route.

### Practice

**Q1 · one line:** Measure the trek line in Web Mercator (EPSG:3857), in km, rounded to 2 decimal places.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
round(trek_line.to_crs(3857).length.iloc[0] / 1000, 2)
```

48.09 km, about 13% too long: the same 1 ÷ cos(latitude) stretch that made areas 28% too large.
</details>

**Q2 · two lines:** Use `peaks_utm` from step 3 to print the distance from Everest to Makalu in km, rounded to 1 decimal place. Everest is row 0 and Makalu row 3.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
everest_to_makalu_m = peaks_utm.geometry.iloc[0].distance(peaks_utm.geometry.iloc[3])
print(round(everest_to_makalu_m / 1000, 1))
```

About 19.5 km. In Lesson B2, shapely gave 0.191 degrees for the same pair, a number with no use on the ground.
</details>

**Q3 · three lines:** How far apart are Dhaulagiri I and Kanchenjunga? Make `by_name = peaks_utm.set_index("name")`, then print the distance in km, rounded to 1 decimal place.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
by_name = peaks_utm.set_index("name")
gap_m = by_name.loc["Kanchenjunga", "geometry"].distance(by_name.loc["Dhaulagiri I", "geometry"])
print(round(gap_m / 1000, 1))
```

About 470.6 km. In Lesson B1 you found 456.7 km for the east–west part only; the straight line is longer, because Kanchenjunga is also about 1° of latitude further south.
</details>

## 7. `set_crs` or `to_crs`? The classic mix-up

Two methods with similar names do two very different jobs:

| Method | What it does | Changes the numbers? | Use it when |
| --- | --- | --- | --- |
| `set_crs(...)` | **labels** the layer: "these numbers are already in this CRS" | no | a layer has no CRS (`.crs` is `None`), for example points from a CSV made without `crs=` |
| `to_crs(...)` | **converts** the numbers into another CRS | yes | the layer has a correct CRS and you need a different one |

A layer without a CRS is called **naive**: GeoPandas does not know what its numbers mean.

Using `set_crs` when you meant `to_crs` is the classic bug. It gives degrees a "metres" label, so every point lands in the wrong place, without any error. GeoPandas refuses to replace an existing CRS unless you add `allow_override=True`; that safety catch exists because of this bug. Below, the bug is made on purpose.

In [ ]:
naive = gpd.GeoDataFrame(peaks, geometry=gpd.points_from_xy(peaks["lon"], peaks["lat"]))
print(naive.crs)

labelled = naive.set_crs("EPSG:4326")
print(labelled.crs)
print(labelled.geometry.iloc[0])

In [ ]:
# THE BUG, ON PURPOSE: "I want metres", so the wrong method is used
wrong = labelled.set_crs(32645, allow_override=True)
print(wrong.geometry.iloc[0])                  # the same numbers, now called metres
print(wrong.to_crs(4326).geometry.iloc[0])     # where GeoPandas now thinks Everest is

In [ ]:
world = gpd.read_file(DATA / "world_countries.geojson")

fig, ax = plt.subplots(figsize=(7, 6))
world.plot(ax=ax, color="lightgrey", edgecolor="white")
labelled.iloc[[0]].plot(ax=ax, color="tab:blue", markersize=40, label="Everest, correct")
wrong.to_crs(4326).iloc[[0]].plot(ax=ax, color="tab:red", markersize=40, label="Everest after set_crs(32645)")
ax.set_xlim(60, 100)
ax.set_ylim(-10, 40)
ax.set_title("Where the set_crs bug puts Everest")
ax.legend(loc="lower left")
plt.show()

`naive.crs` is `None`. `set_crs("EPSG:4326")` added the label without touching the numbers: Everest is still `POINT (86.925 27.9881)`.

The bug kept the same numbers but called them metres: 86.925 m east and 27.988 m north of the zone's origin. Converted back to degrees, "Everest" lands at about 82.5° E and 0.0003° N, in the Indian Ocean near the equator, about 28° of latitude south of the real summit. No error, no warning.

`.iloc[[0]]`, with double brackets, keeps the first row as a one-row table, so it can still be plotted.

The rule: **`set_crs` only for naive layers whose CRS you know; `to_crs` for everything else.**

### Practice

**Q1 · one line:** Label the trek stops correctly: build a naive layer from `trek` with `points_from_xy` (no `crs=`), add `.set_crs("EPSG:4326")` at the end of the same line, and show `.crs`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
gpd.GeoDataFrame(trek, geometry=gpd.points_from_xy(trek["lon"], trek["lat"])).set_crs("EPSG:4326").crs
```

`<Geographic 2D CRS: EPSG:4326>`. This is the right use of `set_crs`: the numbers were longitude and latitude all along; they only lacked a label.
</details>

**Q2 · two lines:** Show that `set_crs` does not move points but `to_crs` does. Print whether Everest's geometry in `labelled` equals the one in `wrong` (`.equals`, Lesson B2). Then print whether it equals the one in `labelled.to_crs(32645)`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(labelled.geometry.iloc[0].equals(wrong.geometry.iloc[0]))
print(labelled.geometry.iloc[0].equals(labelled.to_crs(32645).geometry.iloc[0]))
```

`True`, then `False`. `set_crs` kept the numbers and changed only the label; `to_crs` calculated new numbers.
</details>

**Q3 · three lines:** A colleague sends one GPS point in UTM 45N metres: easting 473366.6, northing 3062515.4. Make a GeoDataFrame from a small table with columns `easting` and `northing`, label it with the right CRS, then convert it to EPSG:4326 and show the geometry. Which trek stop is it?

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
gps = pd.DataFrame({"easting": [473366.6], "northing": [3062515.4]})
gps_gdf = gpd.GeoDataFrame(gps, geometry=gpd.points_from_xy(gps["easting"], gps["northing"]), crs="EPSG:32645")
print(gps_gdf.to_crs(4326).geometry.iloc[0])
```

`POINT (86.7299 27.6866)` (with a few more digits): Lukla. Here `crs="EPSG:32645"` is right, because the numbers really are UTM 45N metres; then `to_crs` converts them to degrees. Label with what the numbers *are*, convert to what you *need*.
</details>

## 8. Buffers in the right unit

A **buffer** is the area within a given distance of a shape: `.buffer(distance)`. Around a point it makes a circle; around a line, a band. Buffers answer questions like "which villages are within 10 km of this landslide?". Lesson B8 uses them fully.

The distance is in the **units of the CRS**. In EPSG:4326, `.buffer(10)` means 10 *degrees*: a circle more than 2,000 km across. GeoPandas warns, but still draws it.

Convert first, then buffer in metres: `layer.to_crs(32645).buffer(10000)` is 10 km. To draw a buffer on a map in degrees, convert the result back with `.to_crs(4326)`.

**Predict:** will the 10-degree circle around Everest fit inside Nepal?

In [ ]:
everest = peaks_gdf[peaks_gdf["name"] == "Everest"]

wrong_buffer = everest.buffer(10)                 # 10 DEGREES: the mistake, shown on purpose
print(wrong_buffer.total_bounds)

right_buffer = everest.to_crs(32645).buffer(10000)   # 10,000 m = 10 km
print(round(right_buffer.area.iloc[0] / 1000000, 1))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

districts.plot(ax=ax1, color="lightgrey", edgecolor="white")
wrong_buffer.plot(ax=ax1, color="tab:red", alpha=0.3)
ax1.set_title("buffer(10) in EPSG:4326: 10 degrees")

districts.plot(ax=ax2, color="lightgrey", edgecolor="white")
right_buffer.to_crs(4326).plot(ax=ax2, color="tab:blue", alpha=0.5)
everest.plot(ax=ax2, color="black", markersize=10)
ax2.set_title("buffer(10000) in UTM 45N: 10 km")
plt.show()

The wrong buffer runs from 76.9° to 96.9° E and from 18.0° to 38.0° N: it swallows Nepal and much of its neighbours. GeoPandas printed a warning, but the code ran.

The right buffer covers 313.7 km². A perfect circle of radius 10 km has an area of π × 10² ≈ 314.2 km². The buffer is a polygon with many straight sides, so it is a tiny bit smaller.

At national scale, the correct 10 km buffer is a small blue disc around Everest. `alpha=0.3` makes a layer partly see-through, so the districts show through it.

### Practice

**Q1 · one line:** Make a 5 km buffer around Lukla (the first row of `trek_gdf`) in UTM 45N, and print its area in km², rounded to 1 decimal place.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
round(trek_gdf.iloc[[0]].to_crs(32645).buffer(5000).area.iloc[0] / 1000000, 1)
```

About 78.4 km², close to π × 5² ≈ 78.5 km². Halving the radius divides the area by four.
</details>

**Q2 · two lines:** Buffer the trek line from step 6 (`trek_line_utm`) by 2 km, and print the area of the band in km², rounded to 1 decimal place.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
trek_band = trek_line_utm.buffer(2000)
print(round(trek_band.area.iloc[0] / 1000000, 1))
```

About 181.5 km²: a band 4 km wide along a 42.4 km line, with round ends. A rough check: 4 × 42.4 ≈ 170 km², plus about 12.6 km² for the two half-circle ends. Where the trail bends, the band overlaps itself, so the true area is a little smaller than that.
</details>

**Q3 · three lines:** Make the mistake on purpose with a small number. Buffer Everest by `0.1` in EPSG:4326, convert the result to UTM 45N, and print its area in km². Compare it with the 10 km buffer.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
degree_buffer = everest.buffer(0.1)
degree_buffer_utm = degree_buffer.to_crs(32645)
print(round(degree_buffer_utm.area.iloc[0] / 1000000, 1))
```

0.1° looks like about 10 km, but the result is not a circle. Its radius is 0.1° of longitude east–west (about 9.8 km) and 0.1° of latitude north–south (about 11.1 km). Its area, about 341.7 km², is neither the 314 km² of a true 10 km circle nor anything you chose. Always buffer in metres.
</details>

## 9. When it breaks: converting a layer that has no CRS

You make points from a CSV, forget `crs=`, and try to convert them to metres. Run the cell.

In [ ]:
naive_trek = gpd.GeoDataFrame(trek, geometry=gpd.points_from_xy(trek["lon"], trek["lat"]))
naive_trek.to_crs(32645)

Read the last line:

`ValueError: Cannot transform naive geometries.  Please set a crs on the object first.`

- **naive geometries** are geometries without a CRS. `to_crs` needs to know where the numbers *come from* before it can calculate where they *go*.
- The message also gives the fix: *set a crs first*.

The right fix is `naive_trek.set_crs("EPSG:4326").to_crs(32645)`, because you know the CSV holds longitude and latitude. Better still, give `crs="EPSG:4326"` when you build the layer, as in Lesson B3.

Do **not** "fix" it with `set_crs(32645)`: the error would go away, and you would have the silent bug from step 7.

**Try it:** fix the second line.

## Mixed review

These exercises combine everything in this lesson. Each one needs 5 to 10 lines. Plan in words first, then write the code.

### Review 1 · The nearest landslides to LS001

Read `practice_landslides.csv`, make points in EPSG:4326, and convert them to UTM 45N. Work out the distance in km from landslide LS001 (row 0) to every landslide, with `.distance()` on the whole GeoSeries. Store it in a column `dist_km`. Show the `id`, `district` and `dist_km` of the four closest records, LS001 itself included.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
slides = pd.read_csv(DATA / "practice_landslides.csv")
slides_gdf = gpd.GeoDataFrame(slides, geometry=gpd.points_from_xy(slides["lon"], slides["lat"]), crs="EPSG:4326")
slides_utm = slides_gdf.to_crs(32645)

ls001 = slides_utm.geometry.iloc[0]
slides_utm["dist_km"] = slides_utm.distance(ls001) / 1000
closest = slides_utm.sort_values("dist_km")
closest[["id", "district", "dist_km"]].head(4)
```

LS001 is 0 km from itself, then LS033 (0.71 km), LS031 (2.58 km) and LS039 (3.88 km), all in Dolakha. `.distance()` on a GeoSeries measures from every point to one geometry at once, in metres because the layer is in UTM.
</details>

### Review 2 · The area of each province

Read `nepal_provinces.geojson`, convert it to UTM 45N, and add an `area_km2` column. Print the provinces sorted from largest to smallest, then the sum of `area_km2`. Does it match the 148,017 km² from the districts?

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
provinces = gpd.read_file(DATA / "nepal_provinces.geojson")
provinces_utm = provinces.to_crs(32645)
provinces_utm["area_km2"] = provinces_utm.area / 1000000

ranked = provinces_utm.sort_values("area_km2", ascending=False)
print(ranked[["province_code", "area_km2"]])
print(round(provinces_utm["area_km2"].sum()))
```

NP06 (Karnali) is the largest at about 30,856 km², and NP02 (Madhesh) the smallest at about 9,588 km². The total is again 148,017 km², because the provinces were made by merging the same districts.
</details>

### Review 3 · Where does UTM 45N stretch the map?

For every district, work out the area in UTM 44N and in UTM 45N, and the difference in percent: (area in 45N − area in 44N) ÷ area in 44N × 100. Store the result in a new column `diff_pct` of a copy of `districts`, made with `districts.copy()`. Print the three districts with the most negative and the three with the most positive difference.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
compare = districts.copy()
area_44 = districts.to_crs(32644).area
area_45 = districts.to_crs(32645).area
compare["diff_pct"] = (area_45 - area_44) / area_44 * 100

ranked = compare.sort_values("diff_pct")
print(ranked[["district", "diff_pct"]].head(3))
print(ranked[["district", "diff_pct"]].tail(3))
```

In the far east (Jhapa, Ilam, Taplejung) zone 45N gives areas about 1.1% *smaller*: zone 44N, far from home there, inflates them. In the far west (Baitadi, Dadeldhura, Kanchanpur) zone 45N gives areas about 1% *larger*. Each zone stretches the land outside it. That is why one zone for all of Nepal is fine for most work, while an equal-area projection is better for precise national figures. `.copy()` keeps the original `districts` unchanged.
</details>

### Review 4 · Peaks near the trail

Make a 10 km buffer around the trek line in UTM 45N. Print its area in km², then the names of the peaks in `peaks_utm` that lie within it, using `.within()` on the whole GeoSeries with the buffer polygon. Finally, draw the buffer, the trek line and the peaks on one map in UTM 45N.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
trail_zone = trek_line_utm.buffer(10000)
zone_polygon = trail_zone.iloc[0]
print(round(zone_polygon.area / 1000000, 1))

near = peaks_utm[peaks_utm.within(zone_polygon)]
print(near["name"].tolist())

fig, ax = plt.subplots(figsize=(6, 6))
trail_zone.plot(ax=ax, color="tab:blue", alpha=0.2)
trek_line_utm.plot(ax=ax, color="tab:blue")
peaks_utm.plot(ax=ax, color="tab:red", marker="^", markersize=50)
ax.set_xlim(zone_polygon.bounds[0] - 5000, zone_polygon.bounds[2] + 5000)
ax.set_ylim(zone_polygon.bounds[1] - 5000, zone_polygon.bounds[3] + 5000)
ax.set_title("Peaks within 10 km of the Everest trek line")
plt.show()
```

The 10 km zone covers about 1,147.4 km², and two peaks lie inside it: Everest and Lhotse. Cho Oyu and Makalu are farther away. The map is in metres, so the axes show eastings and northings. Zooming to the buffer's bounds plus 5 km shows Cho Oyu, top left, outside the zone, and keeps the other peaks off the map.
</details>

## Recap

- A **CRS** gives coordinates their meaning. **Geographic** CRSs use degrees (EPSG:4326, WGS 84); **projected** CRSs use metres (UTM 44N = 32644, UTM 45N = 32645, Web Mercator = 3857).
- Nepal spans UTM zones 44N and 45N, split at 84° E. This course measures all of Nepal in **UTM 45N**; it stretches the far west by about 1%, and an **equal-area** projection is better for precise national totals.
- `to_crs` **converts** coordinates; `set_crs` only **labels** a naive layer. Using `set_crs` when you meant `to_crs` moves data silently to the wrong place.
- Measure after converting: `.area / 1000000` gives km², `.length / 1000` and `.distance() / 1000` give km. In this data Nepal is about 148,017 km², and the trek line 42.37 km.
- Never measure or buffer in degrees or in Web Mercator, which inflates areas in Nepal by about 28%. Buffer in metres: `.to_crs(32645).buffer(10000)` is 10 km.

### Words you learned

| Word | Meaning |
| --- | --- |
| CRS | Coordinate reference system: what coordinate numbers mean, where zero is and in which unit |
| geographic CRS | A CRS with longitude and latitude in degrees, such as EPSG:4326 |
| projected CRS | A CRS that flattens the Earth onto a plane, with x and y in metres |
| map projection | A method of flattening the curved Earth onto a plane; it always distorts something |
| ellipsoid | A slightly flattened sphere used as a model of the Earth's shape |
| datum | An ellipsoid plus how it is tied to the real Earth, such as WGS 84 |
| WGS 84 | The datum and geographic CRS used by GPS; EPSG:4326 |
| EPSG code | A number that identifies a CRS, such as 4326 or 32645 |
| UTM | Universal Transverse Mercator: 60 zones, each 6° wide, with coordinates in metres |
| central meridian | The line of longitude in the middle of a UTM zone, such as 87° E for zone 45 |
| easting / northing | x and y in a projected CRS, in metres |
| reprojecting | Converting coordinates from one CRS to another, with `to_crs` |
| naive layer | A layer whose `.crs` is `None`, so its numbers have no stated meaning |
| equal-area projection | A projection that keeps areas true, such as Asia North Albers Equal Area Conic |
| Web Mercator | EPSG:3857, the projection of web maps; inflates areas away from the equator |
| buffer | The area within a given distance of a shape |
| `alpha` | How opaque a plotted layer is, from 0 (invisible) to 1 (solid) |

## Next

**B5 · Selecting by attribute:** pick the features you need by name, province, area or a box of coordinates. Then show them highlighted on a map of all the districts.

<!-- pygile-nav -->
**PyGILE Course** · lesson 14 of 21 · [Course home](../README.md)  
← Previous: [B3 · GeoDataFrames and file formats: Nepal's districts as a table](B03_GeoDataFrames_and_file_formats.ipynb) · Next: [B5 · Selecting by attribute: finding the districts you need](B05_Selecting_by_attribute.ipynb) →